In [19]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import os
import torchaudio
import json
from torch.nn.utils.rnn import pad_sequence

In [21]:
####### PATHS and VARIABLE CONSTANTS #######
project_root_dir = os.getcwd()
data_root_dir = os.path.join(project_root_dir, "data")
vocabulary_dir = os.path.join(project_root_dir, "vocabulary")
phoneme_to_idx_vocab_filename = "phoneme_to_idx.json"
idx_to_phoneme_vocab_filename = "idx_to_phoneme.json"
train_data_csv = os.path.join(project_root_dir, "train_data.csv")
test_data_csv = os.path.join(project_root_dir, "test_data.csv")

print(f"Project root directory: {project_root_dir}")
print(f"Data root directory: {data_root_dir}")
print(f"Vocabulary directory: {vocabulary_dir}")

Project root directory: /Users/atis/Desktop/MasterArbeit/Code
Data root directory: /Users/atis/Desktop/MasterArbeit/Code/data
Vocabulary directory: /Users/atis/Desktop/MasterArbeit/Code/vocabulary


In [4]:
# detecting the GPU
print("Pytoch version: ", torch.__version__) # check the pytorch version
print("MPS (Metal Performance Shaders) available", torch.backends.mps.is_available()) # check if GPU acceleration is available
print(f"MPS supported for Pytorch version {torch.__version__}:", torch.backends.mps.is_built()) # check if MPS is available for the current pytorch version

# set teh device variable to the GPU, if available
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("GPU acceleration enabled")
else:
    device = torch.device("cpu")
    print("NO GPU! CPU will be used")

Pytoch version:  2.1.2
MPS (Metal Performance Shaders) available True
MPS supported for Pytorch version 2.1.2: True
GPU acceleration enabled


In [16]:
###### CHECK THE OUTPUT OF THE torchaudio.load() method #####
test_waveform, test_sample_rate = torchaudio.load("/Users/atis/Desktop/MasterArbeit/Code/data/TEST/DR2/FDRD1/SA1.WAV")
features = mel_transform(test_waveform)
print("waveform 1 output:")
print("----------------")
print("Type of Waveform: ", type(test_waveform))
print("Shape of Waveform: ", test_waveform.shape)
print("Waveform:", test_waveform)

print("sample_rate 1 output:")
print("-------------------")
print("sample rate type:", type(test_sample_rate))
print("Sample rate:", test_sample_rate)

print("features 1 output:")
print("-------------------")
print("features type:", type(features))
print("features shape:", features.shape)

test_waveform, test_sample_rate = torchaudio.load("/Users/atis/Desktop/MasterArbeit/Code/data/TEST/DR2/FDRD1/SA2.WAV")
features = mel_transform(test_waveform)
print("waveform 2 output:")
print("----------------")
print("Type of Waveform: ", type(test_waveform))
print("Shape of Waveform: ", test_waveform.shape)
print("Waveform:", test_waveform)

print("sample_rate 2 output:")
print("-------------------")
print("sample rate type:", type(test_sample_rate))
print("Sample rate:", test_sample_rate)

print("features 2 output:")
print("-------------------")
print("features type:", type(features))
print("features shape:", features.shape)

waveform 1 output:
----------------
Type of Waveform:  <class 'torch.Tensor'>
Shape of Waveform:  torch.Size([1, 55911])
Waveform: tensor([[ 9.1553e-05,  6.1035e-05,  0.0000e+00,  ..., -6.1035e-05,
          6.1035e-05, -1.2207e-04]])
sample_rate 1 output:
-------------------
sample rate type: <class 'int'>
Sample rate: 16000
features 1 output:
-------------------
features type: <class 'torch.Tensor'>
features shape: torch.Size([1, 80, 350])
waveform 2 output:
----------------
Type of Waveform:  <class 'torch.Tensor'>
Shape of Waveform:  torch.Size([1, 47104])
Waveform: tensor([[ 1.5259e-04,  3.0518e-05, -3.0518e-05,  ..., -1.2207e-04,
          6.1035e-05, -6.1035e-05]])
sample_rate 2 output:
-------------------
sample rate type: <class 'int'>
Sample rate: 16000
features 2 output:
-------------------
features type: <class 'torch.Tensor'>
features shape: torch.Size([1, 80, 295])


In [17]:
####### function for creating the VOCABULARY from the present dataset #######
def build_phoneme_vocab(root_dir, save_dir=None):
    phoneme_set = set()
    
    # iterate over the phoneme files and extract the phonemes
    for dirpath, _, filenames in os.walk(root_dir):
        for file in filenames:
            if file.endswith(".PHN"):
                phn_path = os.path.join(dirpath, file)

                with open(phn_path, "r") as f:
                    for line in f:
                        parts = line.strip().split()
                        phoneme = parts[2]
                        phoneme_set.add(phoneme)
    
    # turn the list of extracted phonemes into a set -> for uniqueness
    phoneme_list = sorted(list(phoneme_set))

    # explicitly define blank
    phoneme_to_idx_vocab = {"<blank>": 0}

    # define the indices for all the other phonemes in the vocabulary
    for i, ph in enumerate(phoneme_list):
        phoneme_to_idx_vocab[ph] = i + 1

    # create a vocabulary where the indices are the keys and the phonemes are the values
    idx_to_phoneme_vocab = {i: ph for ph, i in phoneme_to_idx_vocab.items()}

    # save the created vocabularies in json files
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)

        with open(os.path.join(save_dir, "phoneme_to_idx.json"), "w") as f:
            json.dump(phoneme_to_idx_vocab, f, indent=4)

        with open(os.path.join(save_dir, "idx_to_phoneme.json"), "w") as f:
            json.dump(idx_to_phoneme_vocab, f, indent=4)

    return phoneme_to_idx_vocab, idx_to_phoneme_vocab

In [18]:
####### BUILDING the VOCABULARY #######
if (os.path.exists(os.path.join(vocabulary_dir, phoneme_to_idx_vocab_filename)) == False or 
    os.path.exists(os.path.join(vocabulary_dir, idx_to_phoneme_vocab_filename)) == False):
    phoneme_to_idx_vocab, idx_to_phoneme_vocab = build_phoneme_vocab(root_dir=data_root_dir,
                                                                     save_dir=vocabulary_dir)
    print(f"The following vocabulary files were created:\n{phoneme_to_idx_vocab_filename}\n{idx_to_phoneme_vocab_filename}")
else:
    with open(os.path.join(vocabulary_dir, phoneme_to_idx_vocab_filename), "r") as file:
        phoneme_to_idx_vocab = json.load(file)
    with open(os.path.join(vocabulary_dir, idx_to_phoneme_vocab_filename), "r") as file:
        idx_to_phoneme_vocab = json.load(file)
        
        # convert keys back to int
        idx_to_phoneme_vocab = {int(k): v for k, v in idx_to_phoneme_vocab.items()}
    print(f"the vocabulary files already exist in the vocabulary directory:\n{phoneme_to_idx_vocab_filename}\n{idx_to_phoneme_vocab_filename}")

The following vocabulary files were created:
phoneme_to_idx.json
idx_to_phoneme.json


In [30]:
class TIMITDataset(Dataset):
    def __init__(self,
                 csv_file: str,
                 train_or_test: str,
                 root_dir: str,
                 phoneme_to_idx_vocab: dict, 
                 transform: any = None):
        """Constructor of the class
        
        Parameters
        ----------
        csv_file: str
            the path to the csv file, containing the informations about the dataset

        train_or_test: str
            to select the train and test parts in the dataset, if they are present 
        
        root_dir : str
            the root directory of the dataset

        phoneme_to_idx_vocab: dict
            the vocabulary of the phonemes

        transform: any
            the transformation object to apply on the raw waveform => e.g. mel spectrogram
        """
        
        self.df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.phoneme_to_idx_vocab = phoneme_to_idx_vocab
        self.transform = transform


        # check if the dataframe contains only train or test data
        if (self.df["test_or_train"] == train_or_test).all():
            print(f"All samples are from the {train_or_test} ")
        else:
            print(f"Filtering dataframe to only include {train_or_test} samples")
            self.df = self.df.loc[self.df["test_or_train"] == train_or_test]

        # get the root directory of the dataset
        self.root_dir = os.path.join(self.root_dir, "data")
        
        # create a base name column once
        self.df["base_name"] = self.df["filename"].str.split(".").str[0]

        # build the a sample list where each element is a dictionary.
        # keys -> base_name, audio_path, phoneme_path
        # values -> base name, path to the audio file, path to the phoneme file
        # build the sample list once
        self.samples = []
        grouped = self.df.groupby("base_name")

        for base_name, group in grouped:
            audio_rows = group[
                (group["is_audio"] == True) &
                (group["filename"].str.endswith(".WAV"))
            ]
            phoneme_rows = group[group["is_phonetic_file"] == True]

            # check for anomalies in the dataset
            if len(audio_rows) != 1 or len(phoneme_rows) != 1:
                continue
            
            # define the full path to the audio files
            audio_path = os.path.join(self.root_dir, audio_rows.iloc[0]["path_from_data_dir"])
            phoneme_path = os.path.join(self.root_dir, phoneme_rows.iloc[0]["path_from_data_dir"])

            # adding the dictionaries to the samples list
            self.samples.append({
                "base_name": base_name,
                "audio_path": audio_path,
                "phoneme_path": phoneme_path
            })



    def __len__(self) -> int:
        """Returns the length of the dataset
        
        Parameters
        ----------
        None
        
        Returns
        -------
        int: 
            the length of the dataset
        """
        return len(self.samples)


    def __getitem__(self, 
                    idx: int) -> tuple:
        """Returns one element from the dataset (feature, label)
        
        Parameters
        ----------
        idx: int
            the index of the element in the dataset
        
        Returns
        -------
        tuple:
            1. element: the melspectrogram transformed tensor of the audio file
            2. element: the phoneme sequence of the the audio file, as integers from the vocabulary
        """
        # get the base string of the filename
        my_sample = self.samples[idx]

        # build the names of the audio and phoneme files
        path_audio_file = my_sample["audio_path"]
        path_phoneme_file = my_sample["phoneme_path"]

        # turning the audio file into a tensor
        if os.path.exists(path_audio_file):
            waveform, sample_rate = torchaudio.load(path_audio_file)
        else:
            raise Exception(f"Audio file {path_audio_file} does no exist")

        phoneme_ids = []

        # turning the phonemes into a list of integer numbers
        if os.path.exists(path_phoneme_file):
            with open(path_phoneme_file, "r") as f:
                for line in f:
                    phoneme_str = (line.strip().split())[-1]
                    try:
                        phoneme_ids.append(self.phoneme_to_idx_vocab[phoneme_str])
                    except KeyError as k:
                        raise KeyError(f"No such key in the vocabulary as {phoneme_str}")
        else:
            raise Exception(f"Phoneme file {path_phoneme_file} does not exist")
        
        # turning the list of integers into a 1D tensor of integers
        phoneme_labels = torch.tensor(phoneme_ids, dtype=torch.long)
        
        # applying the melspectrogram transform on the samples
        if self.transform is not None:
            mel_spec = self.transform(waveform)
            mel_spec = mel_spec.squeeze(0).transpose(0,1) # (T, nmels) -> T = number of windows
            mel_spec = torch.log(mel_spec + 1e-6) #stabilizes training and compresses dynamic range
        else:
            mel_spec = waveform
        return mel_spec, phoneme_labels
        

        

In [31]:
####### Transformations for the dataset ######
# The Mel spectrogram does the following:
#1. Split signal into windows
#2. Apply Fourier Transform (FFT)
#3. Convert frequencies to Mel scale
#4. Output energy per Mel band
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_fft=400, # FFT window size => how many samples are used for frequency analysis
    hop_length=160, # step size between windows => at 16kHz a hop length of 160 means a stepsize of 10ms
    win_length=400, # Actual window size applied before FFT => at 16kHz a window size of 400 means 25ms window length
    n_mels=80 # Number of frequency bins after Mel conversion => number of frequency elements in each window AFTER conversion
)

In [32]:
####### Collate functions for the dataloaders #####
def collate_fn_ctc(batch: list) -> tuple:
    """
    Collate function for variable-length TIMIT samples.

    Parameters
    ----------
    batch : list
        List of tuples (features, labels):
        - features has shape [T, 80]
        - labels has shape [L]

    Returns
    -------
    tuple: 
        1. element: padded_features : torch.Tensor
                    Shape [B, T_max, 80]

        2. element: feature_lengths : torch.Tensor
                    Shape [B]

        3. element: padded_labels : torch.Tensor
                    Shape [B, L_max]

        4. element: label_lengths : torch.Tensor
                    Shape [B]
    """

    # unzip batch into two tuples
    feature_list, label_list = zip(*batch)

    # store original lengths before padding
    feature_lengths_original = torch.tensor(
        [x.shape[0] for x in feature_list],
        dtype=torch.long
    )

    label_lengths_original = torch.tensor(
        [y.shape[0] for y in label_list],
        dtype=torch.long
    )

    # pad features along time dimension
    # each feature tensor has shape [T, 80]
    # result: [B, T_max, 80]
    padded_features = pad_sequence(
        feature_list,
        batch_first=True,
        padding_value=0.0
    )

    concatenated_labels = torch.cat(label_list, dim=0)

    return padded_features, feature_lengths_original, concatenated_labels, label_lengths_original

In [33]:
####### datasets and DATALOADERS #######
train_dataset = TIMITDataset(
    csv_file=train_data_csv,
    train_or_test="TRAIN",
    root_dir=project_root_dir,
    phoneme_to_idx_vocab=phoneme_to_idx_vocab,
    transform=mel_transform
)

test_dataset = TIMITDataset(
    csv_file=test_data_csv,
    train_or_test="TEST",
    root_dir=project_root_dir,
    phoneme_to_idx_vocab=phoneme_to_idx_vocab,
    transform=mel_transform
)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=200,
    shuffle=True,
    collate_fn=collate_fn_ctc
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=60,
    shuffle=False,
    collate_fn=collate_fn_ctc
)

Filtering dataframe to only include TRAIN samples
Filtering dataframe to only include TEST samples


In [26]:
####### TESTING THE SHAPES OF ONE BATCH ######
batch = next(iter(train_dataloader))
features, feature_lengths, labels, label_lengths = batch

print(features.shape)
print(feature_lengths.shape)
print(labels.shape)
print(label_lengths.shape)

torch.Size([200, 665, 80])
torch.Size([200])
torch.Size([8563])
torch.Size([200])
